<a href="https://colab.research.google.com/github/440g/painkiller/blob/Model-evaluatioin/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model training and validation
* Selection of multiple machine learning algorithms.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

X_train = pd.read_csv('/content/drive/MyDrive/painkiller/datasets/X_train.csv')
#파일 맨 앞에 의미없는 인덱스 값이 존재해 제거
X_train = X_train.drop(columns=['Unnamed: 0'])
y_train = pd.read_csv('/content/drive/MyDrive/painkiller/datasets/y_train.csv')
X_val = pd.read_csv('/content/drive/MyDrive/painkiller/datasets/X_val.csv')
X_val = X_val.drop(columns=['Unnamed: 0'])
y_val = pd.read_csv('/content/drive/MyDrive/painkiller/datasets/y_val.csv')

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import KFold, GridSearchCV

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = "f1"

models = {}

###1. Decision Tree

In [ ]:
model = DecisionTreeClassifier(random_state=42)


param_grid = {
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 10, 20],
    "ccp_alpha": [0.0, 0.01],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Decision Tree"] = grid_search.best_estimator_

Best parameters:  {'ccp_alpha': 0.0, 'max_depth': 5, 'min_samples_split': 2}
Best CV score: 0.635988


###2. Bagging

In [ ]:
model = BaggingClassifier(estimator=DecisionTreeClassifier(),
                         n_jobs=-1,
                         random_state=42)


param_grid = {
    "n_estimators": [25, 50]
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Bagging"] = grid_search.best_estimator_

Best parameters:  {'n_estimators': 50}
Best CV score: 0.623175


###3. Random Forest

In [ ]:
model = RandomForestClassifier(n_jobs=-1, random_state=42)


param_grid = {
    "n_estimators": [25, 50],

    # 기본값인 "sqrt" 보다 더 적은 max_features를 사용할 때 모델 성능 비교
    "max_features": ["sqrt", "log2"],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Random Forest"] = grid_search.best_estimator_

Best parameters:  {'max_features': 'sqrt', 'n_estimators': 50}
Best CV score: 0.630577


###4. AdaBoost

In [ ]:
model = AdaBoostClassifier(random_state=42)


param_grid = {
    #과적합 방지를 위한 비교적 낮은 max_depth의 트리와 성능을 위한 높은 max_depth의 결과를 비교
    "estimator": [DecisionTreeClassifier(max_depth=3), DecisionTreeClassifier(max_depth=6)],
    "learning_rate": [0.1, 1.0],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["AdaBoost"] = grid_search.best_estimator_

Best parameters:  {'estimator': DecisionTreeClassifier(max_depth=6), 'learning_rate': 0.1}
Best CV score: 0.648480


###5. Gradient Boosting

In [ ]:
model = GradientBoostingClassifier(
                                  random_state=42)

param_grid = {
    "max_depth": [3, 6],

    # learning_rate = 0.0을 base line으로 이용해 비교
    "learning_rate": [0.0, 0.1],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Gradient Boosting"] = grid_search.best_estimator_

Best parameters:  {'learning_rate': 0.1, 'max_depth': 3}
Best CV score: 0.652684


###6. XG Boost

In [ ]:
model = XGBClassifier(learning_rate=0.1,
                     n_jobs=-1,
                     random_state=42)

param_grid = {
    "n_estimators": [25, 50],

    # 데이터의 feature가 많기 때문에 L1과 L2 정규화를 이용해 더 좋은 성능이 나오는 지 탐색
    "reg_alpha": [0, 0.1],
    "reg_lambda": [0, 0.1],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["XGBoost"] = grid_search.best_estimator_

Best parameters:  {'n_estimators': 50, 'reg_alpha': 0, 'reg_lambda': 0.1}
Best CV score: 0.654553


###7. Light GBM

In [ ]:
model = LGBMClassifier(data_sample_strategy="goss",
                      top_rate=0.2,
                      other_rate=0.1,
                      force_col_wise=True,
                      verbosity=0,
                      n_jobs=-1,
                      random_state=42)

#Light GBM 모델이 특수기호가 있으면 읽지 못해 없애는 과정이 필요함
X_train_lgbm = X_train.copy()
X_val_lgbm = X_val.copy()
X_train_lgbm.columns = X_train_lgbm.columns.str.replace(r"[^\w]", "_", regex = True)
X_val_lgbm.columns = X_val_lgbm.columns.str.replace(r"[^\w]", "_", regex = True)

param_grid = {
    "n_estimators": [25, 50],

    # 데이터의 feature가 많기 때문에 L1과 L2 정규화를 이용해 더 좋은 성능이 나오는 지 탐색
    "reg_alpha": [0, 0.1],
    "reg_lambda": [0, 0.1],

    # feature를 bundle로 처리할 때 달라지는 성능과 속도를 비교
    "enable_bundle": [True, False]
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True)
grid_search.fit(X_train_lgbm, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["LightGBM"] = grid_search.best_estimator_

Best parameters:  {'enable_bundle': True, 'n_estimators': 25, 'reg_alpha': 0, 'reg_lambda': 0}
Best CV score: 0.641982


In [ ]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

for _name, _model in models.items():
    y_pred = _model.predict(X_val)

    f1 = f1_score (y_val["y"], y_pred)
    accuracy = accuracy_score(y_val["y"], y_pred)
    AUC = roc_auc_score (y_val["y"], y_pred)
    Main_Evaluation = (f1 + accuracy + AUC)/3
    print("{:>17}: f1 = {:.4f} | accuracy = {:.4f} | AUC = {:.4f} | Main Evaluation = {:.4f}".format(
        _name, f1, accuracy, AUC, Main_Evaluation))



    Decision Tree: f1 = 0.6286 | accuracy = 0.6087 | AUC = 0.6095 | Main Evaluation = 0.6156
          Bagging: f1 = 0.6297 | accuracy = 0.6388 | AUC = 0.6386 | Main Evaluation = 0.6357
    Random Forest: f1 = 0.6298 | accuracy = 0.6423 | AUC = 0.6420 | Main Evaluation = 0.6380
         AdaBoost: f1 = 0.6395 | accuracy = 0.6543 | AUC = 0.6539 | Main Evaluation = 0.6492
Gradient Boosting: f1 = 0.6534 | accuracy = 0.6627 | AUC = 0.6624 | Main Evaluation = 0.6595
          XGBoost: f1 = 0.6422 | accuracy = 0.6499 | AUC = 0.6498 | Main Evaluation = 0.6473
         LightGBM: f1 = 0.6330 | accuracy = 0.6423 | AUC = 0.6421 | Main Evaluation = 0.6391


# Test set preprocessing

In [ ]:
test_df = pd.read_csv('/content/drive/MyDrive/painkiller/datasets/test.csv')
df = pd.read_csv('/content/drive/MyDrive/painkiller/datasets/train.csv')

In [ ]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9515 entries, 0 to 9514
Data columns (total 47 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            9515 non-null   int64  
 1   n_tokens_title                8562 non-null   float64
 2   n_tokens_content              8540 non-null   float64
 3   n_unique_tokens               8574 non-null   float64
 4   n_non_stop_words              8554 non-null   float64
 5   n_non_stop_unique_tokens      8600 non-null   float64
 6   num_hrefs                     8571 non-null   float64
 7   num_self_hrefs                8570 non-null   float64
 8   num_imgs                      8587 non-null   float64
 9   num_videos                    8485 non-null   float64
 10  average_token_length          8540 non-null   float64
 11  num_keywords                  8583 non-null   float64
 12  kw_min_min                    8591 non-null   float64
 13  kw_

In [ ]:
#train set의 전처리 과정을 그대로 따름
##전처리 과정에서 train set의 전처리에서 썼던 값(e.g. median, mean)이 필요한데 따로 작성하다보니 똑같이 적용이 어려워서 코드를 합친 후에 수정이 조금 필요할 것 같음

num_cols = test_df.select_dtypes(include=['float64', 'int64']).columns.drop(['id'])
test_df[num_cols] = test_df[num_cols].fillna(df[num_cols].median())
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

mean_cols = [
    'global_sentiment_polarity', 'global_rate_positive_words',
    'rate_negative_words', 'avg_negative_polarity', 'min_negative_polarity',
    'title_sentiment_polarity', 'title_subjectivity'
]
for col in mean_cols:
    test_df[col] = test_df[col].mask(test_df[col].isna(), df[col].mean())


test_df['data_channel']=test_df['data_channel'].fillna('unknown')
test_df['weekday']=test_df['weekday'].fillna('unknown')

test_df = pd.get_dummies(test_df, columns=['data_channel', 'weekday'], drop_first=True)

In [ ]:
import numpy as np

clip_cols = [
    'n_tokens_content', 'num_hrefs', 'num_self_hrefs',
    'kw_avg_avg']

for col in clip_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    test_df[col] = test_df[col].clip(lower, upper)

    test_df['num_videos'] = test_df['num_videos'].clip(upper=6)
    test_df['num_imgs'] = test_df['num_imgs'].clip(upper=20)


test_df = test_df.drop(columns=['kw_min_min'])


test_df['average_token_length'] = test_df['average_token_length'].replace(0, np.nan)
test_df['average_token_length'] = test_df['average_token_length'].fillna(df['average_token_length'].median())


test_df['kw_avg_min'] = test_df['kw_avg_min'].replace(-1, np.nan)
test_df['kw_avg_min'] = test_df['kw_avg_min'].fillna(df['kw_avg_min'].median())
test_df['kw_avg_min'] = np.log1p(test_df['kw_avg_min'].clip(lower=0))


test_df['kw_min_max'] = test_df['kw_min_max'].replace(0, np.nan)
test_df['kw_min_max'] = test_df['kw_min_max'].fillna(df['kw_min_max'].median())


test_df['kw_min_avg'] = test_df['kw_min_avg'].replace(-1, np.nan)
test_df['kw_min_avg'] = test_df['kw_min_avg'].replace(0, np.nan)
test_df['kw_min_avg'] = test_df['kw_min_avg'].fillna(df['kw_min_avg'].median())


log_transform_cols = ['num_hrefs', 'kw_max_min', 'kw_min_max', 'kw_max_avg','self_reference_min_shares', 'self_reference_max_shares', 'self_reference_avg_sharess']

for col in log_transform_cols:
  test_df[col] = np.log1p(test_df[col])

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = test_df.select_dtypes(include=['float64', 'int64']).columns.drop(['id'])


scaler = StandardScaler()

## scaling 할 때도 test set말고 train set의 기준을 똑같이 적용해야 해서 preprocessing과 코드를 하나로 합친 후 수정이 필요함
#df[num_cols] = scaler.fit_transform(df[num_cols])
#test_df[num_cols] = scaler.transform(test_df[num_cols])

test_df[num_cols] = scaler.fit_transform(test_df[num_cols])

test_preprocessed = test_df.drop(columns=['id'])

test_preprocessed.to_csv('/content/drive/MyDrive/painkiller/datasets/test_preprocessed.csv', index=False)